# Tutorial: Palette catalog summaries

## Audience
- Analysts or developers who need quick summaries from a color catalog CSV.

## Prerequisites
- Basic Python syntax (functions, loops, dictionaries).
- Familiarity with CSV files.

## Learning goals
- Read UTF-8 CSV data by header names.
- Normalize and group palette rows by primary color.
- Produce sorted summary objects for downstream use.


## Outline
1. Define `summarize_palettes(csv_path)`.
2. Run it on `/inputs/colors.csv`.
3. Review edge-case checks (duplicates, order changes, commas, empty input).


In [ ]:
from __future__ import annotations

import csv
from collections import defaultdict

def summarize_palettes(csv_path):
    groups = defaultdict(lambda: {'count': 0, 'product_types': set()})

    with open(csv_path, 'r', encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            primary = (row.get('Primary (Hex)') or '').strip().upper()
            product_type = (row.get('Product Type') or '').strip()

            groups[primary]['count'] += 1
            groups[primary]['product_types'].add(product_type)

    summary = [
        {
            'primary': primary,
            'count': data['count'],
            'product_types': sorted(data['product_types']),
        }
        for primary, data in groups.items()
    ]

    summary.sort(key=lambda item: (-item['count'], item['primary']))
    return summary


## Run the summary on the provided catalog
The next cell assigns `palette_summary` from the public catalog path.


In [ ]:
palette_summary = summarize_palettes('/inputs/colors.csv')
palette_summary_count = len(palette_summary)


## Exercises
- Add a filter that returns only groups with `count >= 2`.
- Extend the summary with a sorted list of row numbers (`No`) per primary color.
- Write a small test for rows where `Product Type` is blank after stripping whitespace.


In [ ]:
import tempfile

sample_csv = (
    'Product Type,Primary (Hex),Notes\n'
    '"Widgets, Deluxe",#abc123,x\n'
    'Tool,#ABC123,y\n'
    'Tool,#def000,z\n'
    ' Tool , #DEF000 ,z2\n'
)

with tempfile.NamedTemporaryFile('w', encoding='utf-8', newline='', delete=False) as tmp:
    _ = tmp.write(sample_csv)
    tmp_path = tmp.name

test_summary = summarize_palettes(tmp_path)
assert test_summary == [
    {'primary': '#ABC123', 'count': 2, 'product_types': ['Tool', 'Widgets, Deluxe']},
    {'primary': '#DEF000', 'count': 2, 'product_types': ['Tool']},
]

with tempfile.NamedTemporaryFile('w', encoding='utf-8', newline='', delete=False) as tmp2:
    _ = tmp2.write('Product Type,Primary (Hex)\n')
    tmp2_path = tmp2.name

assert summarize_palettes(tmp2_path) == []
